# Week 2 Tue: pandas DataFrames, indexing, joins, and missing values

Estimated time: 8 hours

## Why this matters
pandas DataFrames, indexing, joins, and missing values is part of real quant work inside data and linear algebra: numpy, pandas, visualization, diversification, and sql basics research, trading, or risk workflows.

## Session plan
- Session 1 (45 min): Learn DataFrame structure and indexing.
- Session 2 (55 min): Clean missing values and basic data problems.
- Session 3 (55 min): Join and reshape simple market tables.
- Session 4 (55 min): Code drills and small cleaning exercises.
- Session 5 (30 min): Reflection and interview recap.

## Concept notes
### DataFrames as structured market tables
A DataFrame is like a spreadsheet with labeled rows and columns.

Technical note: pandas DataFrames provide indexing, grouping, joining, missing-value handling, and time-series functionality used heavily in research workflows.

Finance example: You might store dates, prices, returns, sectors, and volumes in one table and transform it before modeling.

### Missing values and bad data
Real market data often has gaps, duplicate records, or inconsistent labels.

Technical note: Cleaning decisions matter because dropped rows, forward fills, or wrong joins can change backtest outputs.

Finance example: A missing price on one date can distort return calculations unless handled explicitly.

### Joins and alignment
A join combines tables using a common key such as date or ticker.

Technical note: Misaligned joins are a common source of subtle leakage and incorrect analysis in finance data pipelines.

Finance example: Joining returns with a factor table on the wrong date can accidentally use future information.

## Continuity prompt
- What from yesterday's topic still needs reinforcement?
- What should tomorrow build on from today?

## Real-world dataset for today's labs
Use `curriculum/datasets/real_market_prices.csv` as your default market panel (SPY, QQQ, TLT, GLD).


## Code Lab 1: Build and inspect a price table

In [ ]:
import pandas as pd

prices = pd.DataFrame(
    {
        "date": pd.date_range("2026-01-01", periods=5, freq="D"),
        "asset_a": [100, 101, None, 103, 104],
        "asset_b": [50, 49, 50, 51, 52],
    }
).set_index("date")
print(prices)
print("\nMissing values:")
print(prices.isna().sum())


## Code Lab 2: Clean and compute returns

In [ ]:
cleaned = prices.ffill()
returns = cleaned.pct_change().dropna()
print(cleaned)
print("\nReturns:")
print(returns.round(4))


## Code Lab 3: Join with simple metadata

In [ ]:
metadata = pd.DataFrame(
    {
        "ticker": ["asset_a", "asset_b"],
        "sector": ["equity", "commodity_proxy"],
    }
)
melted = cleaned.reset_index().melt(id_vars="date", var_name="ticker", value_name="price")
merged = melted.merge(metadata, on="ticker", how="left")
print(merged.head())


## Practice recap
- Why is missing data dangerous in return calculations?
  Suggested answer: Because missing values can create false jumps, incorrect percentage changes, or unintended row drops if handled casually.
- What is one common join mistake in finance data?
  Suggested answer: Joining on the wrong date key and accidentally aligning today's feature with tomorrow's outcome.
- Why do labels matter in pandas?
  Suggested answer: Because labeled indices and columns make transformations safer and easier to interpret than raw position-only arrays.

## Interview drill
- Q: What is the difference between a DataFrame and a NumPy array in practice?
  A: A DataFrame has labels and richer data-wrangling features, while a NumPy array is more lightweight and numerically focused.

## 4+ Hour Completion Roadmap

Use this minimum structure to turn today's notebook into a serious study session:

- Phase 1 (60 min): Read concept notes and rewrite the core idea from memory.
- Phase 2 (70 min): Complete and extend all code labs with at least one variation.
- Phase 3 (65 min): Do formula retrieval, scenario analysis, and error-log updates.
- Phase 4 (65 min): Write a mini memo and practice interview responses aloud.

## Formula Rewrite Drill

In [ ]:
import pandas as pd

formula_drill = pd.DataFrame(
    [
        {"formula": "E[X] = sum_i p_i x_i", "from_memory": "", "term_explanations": ""},
        {"formula": "Var(X) = E[(X - E[X])^2]", "from_memory": "", "term_explanations": ""},
        {"formula": "W_t = W_0 * product(1 + r_t)", "from_memory": "", "term_explanations": ""},
        {"formula": "Topic formula for: pandas DataFrames, indexing, joins, and missing values", "from_memory": "", "term_explanations": ""},
    ]
)
print(formula_drill)


## Real Market Data Lab (Useful From Day 1)

This section uses a local CSV snapshot of real market prices so the notebook remains reproducible.

Dataset: `curriculum/datasets/real_market_prices.csv`
Symbols: SPY, QQQ, TLT, GLD

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("curriculum/datasets/real_market_prices.csv")
market = pd.read_csv(data_path, parse_dates=["date"]).sort_values(["date", "symbol"])

print("Rows:", len(market))
print("Date range:", market["date"].min().date(), "to", market["date"].max().date())
print("Symbols:", sorted(market["symbol"].unique()))
print(market.head())

prices = market.pivot(index="date", columns="symbol", values="close").dropna()
returns = prices.pct_change().dropna()

summary = pd.DataFrame(
    {
        "ann_return": returns.mean() * 252,
        "ann_vol": returns.std() * (252 ** 0.5),
        "max_drawdown": ((1 + returns).cumprod().div((1 + returns).cumprod().cummax()) - 1).min(),
    }
).sort_index()
print("\nRisk/return summary:")
print(summary.round(4))

cum = (1 + returns).cumprod()
cum.plot(figsize=(10, 5), title="Cumulative growth (base 1.0)")
plt.ylabel("Growth of $1")
plt.tight_layout()
plt.show()


## Real-World Takeaway Prompt

Write 5-8 lines for today's topic (pandas DataFrames, indexing, joins, and missing values):

1. Which symbol looked most volatile and why that matters.
2. Which pair looked most diversifying.
3. One realistic portfolio/risk decision you could make from this table.

## Interview Question + Python Solution Drill

Use this format for interview preparation:

1. State the question clearly.
2. Explain your approach in plain language.
3. Write Python code on real market data.
4. Interpret one risk caveat in words.

**Suggested data-source ladder**
- Source 1: yfinance pull (fresh market data)
- Source 2: Robinhood-style CSV export (if available locally)
- Source 3: local snapshot `curriculum/datasets/real_market_prices.csv` (reproducible fallback)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

prices = None
source_used = ""
USE_YFINANCE = False  # set True when you want a live market pull

try:
    import yfinance as yf
except ImportError:
    yf = None

if USE_YFINANCE and yf is not None:
    downloaded = yf.download(
        ["SPY", "QQQ", "TLT", "GLD"],
        period="2y",
        interval="1d",
        auto_adjust=True,
        progress=False,
    )
    if not downloaded.empty:
        if isinstance(downloaded.columns, pd.MultiIndex):
            if "Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Close"].copy()
            elif "Adj Close" in downloaded.columns.get_level_values(0):
                prices = downloaded["Adj Close"].copy()
        else:
            prices = downloaded.rename(columns=str.upper)
        source_used = "yfinance"

robinhood_export = Path("curriculum/datasets/robinhood_export.csv")
if prices is None and robinhood_export.exists():
    rh = pd.read_csv(robinhood_export, parse_dates=["date"])
    prices = rh.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "robinhood_export_csv"

if prices is None:
    local = pd.read_csv(
        Path("curriculum/datasets/real_market_prices.csv"),
        parse_dates=["date"],
    )
    prices = local.pivot(index="date", columns="symbol", values="close").sort_index()
    source_used = "local_snapshot_csv"

prices = prices.dropna(how="all").ffill().dropna()
returns = prices.pct_change().dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

annualized_return = returns.mean() * 252
annualized_vol = returns.std() * np.sqrt(252)
sharpe_proxy = (annualized_return - 0.03) / annualized_vol

print("Source used:", source_used)
print("\nAnnualized return:")
print(annualized_return.round(4))
print("\nAnnualized volatility:")
print(annualized_vol.round(4))
print("\nSharpe proxy (rf=3%):")
print(sharpe_proxy.round(3))
print("\nRecent log returns:")
print(log_returns.tail().round(4))


## Scenario Analysis Drill

In [ ]:
import pandas as pd

scenarios = pd.DataFrame(
    [
        {"scenario": "base_case", "assumption": "", "expected_effect": ""},
        {"scenario": "bull_case", "assumption": "", "expected_effect": ""},
        {"scenario": "stress_case", "assumption": "", "expected_effect": ""},
    ]
)
print("Topic:", 'pandas DataFrames, indexing, joins, and missing values')
print(scenarios)


## Closed-Book Retrieval and Error Log

In [ ]:
import pandas as pd

retrieval_scorecard = pd.DataFrame(
    [
        {"prompt": "Explain today's concept in 3 lines", "score_0_to_5": None, "notes": ""},
        {"prompt": "Write one key formula without notes", "score_0_to_5": None, "notes": ""},
        {"prompt": "Give one realistic failure mode", "score_0_to_5": None, "notes": ""},
        {"prompt": "Connect to one trading/risk decision", "score_0_to_5": None, "notes": ""},
    ]
)

error_log = pd.DataFrame(
    [
        {
            "concept": "",
            "mistake": "",
            "correction": "",
            "next_review_date": "",
        }
    ]
)
print("Retrieval scorecard template:")
print(retrieval_scorecard)
print("\nError log template:")
print(error_log)


## Final 30-Minute Deliverable

Write a short memo (150-250 words) with this structure:

1. Core idea of pandas DataFrames, indexing, joins, and missing values in plain language.
2. One technical detail or formula and why it matters.
3. One practical quant use case.
4. One limitation or failure mode and how you would detect it.